# Experiment Infrastructure

## EventLogBuilder

In [1]:
from collections.abc import Mapping, Sequence
import datetime as dt
from typing import Any
import xml.etree.ElementTree as ET

TIMESTAMP = "time:timestamp"
ACTIVITY = "concept:name"
TASK_ID = "concept:instance"
TRANSITION = "lifecycle:transition"


class EventLogBuilder:

    def __init__(self) -> None:
        self._traces: list[dict[str, Any]] = []
        self._current_trace: dict[str, Any] | None = None
        self._current_activity : str | None = None
        self._current_task : str | None = None
        self._current_time : dt.datetime | None = None

    def add_trace(self, case_attributes = dict()):
        self._current_trace = {"case_attributes": case_attributes, "events": []}
        self._traces.append(self._current_trace)
        return self

    def add_task(
        self,
        activity_name: str
    ):
        if self._current_trace is None:
            raise ValueError("A trace must be added before adding activities.")

        self._current_activity = activity_name
        self._current_task = activity_name + f'_{len(self._traces)}_{len(self._current_trace['events'])}'
        
        return self

    def _event(self, timestamp, transition, attributes: Mapping[str, Any] = None):
        event: dict[str, Any] = {
            ACTIVITY : self._current_activity, 
            TASK_ID : self._current_task , 
            TIMESTAMP: dt.datetime.fromisoformat(timestamp), 
            TRANSITION : transition 
        }
        if attributes:
            event.update(attributes)
        self._current_trace["events"].append(event)
        return self

    def start(self, start_timestamp, attributes: Mapping[str, Any] = None):
        return self._event(start_timestamp, "start", attributes)

    def complete(self, end_timestamp, attributes: Mapping[str, Any] = None):
        return self._event(end_timestamp, "complete", attributes)

    def set_current_time(self, timestamp):
        self._current_time = dt.datetime.fromisoformat(timestamp)
        return self

    def get_current_time(self):
        if self._current_time is not None:
            return self._current_time
        else:
            # Latest timestamp of all events in all traces
            return max((event[TIMESTAMP] for trace in self._traces for event in trace["events"]), default=None)



    def to_xes(self, log_name: str = "example_log") -> str:
        if not self._traces:
            raise ValueError("Cannot serialize an empty event log.")
        
        log = ET.Element(
            "log",
            {
                "xes.version": "2.0",
                # "xes.features": "nested-attributes",
                # "openxes.version": "1.0RC7",
                "xmlns": "http://www.xes-standard.org/",
            },
        )
        ET.SubElement(log, "string", {"key": "concept:name", "value": str(log_name)})

        trace = self._traces[-1]
        case_values = dict(trace["case_attributes"])
        trace_name = str(case_values.get("concept:name", 'trace_1'))

        trace_xml = ET.SubElement(log, "trace")
        ET.SubElement(trace_xml, "string", {"key": "concept:name", "value": trace_name})
        for key, value in case_values.items():
            if key == "concept:name":
                continue
            _xes_element(trace_xml, key, value)

        for event in sorted(trace["events"], key=lambda x: x[TIMESTAMP]):
            event_xml = ET.SubElement(trace_xml, "event")
            for key, value in event.items():
                _xes_element(event_xml, key, value)

        ET.indent(log, space="\t", level=0)
        return '<?xml version="1.0" encoding="UTF-8"?>\n' + ET.tostring(log, encoding="unicode")

def is_collection(obj):
    return hasattr(obj, "__iter__") and not isinstance(obj, str)

def _xes_element(parent : ET.Element, key, value):
    if is_collection(value):
        element = ET.SubElement(parent, 'list', {"key": str(key)})
        values = ET.SubElement(element, 'values')
        for subvalue in value:
            _xes_element(values, key, subvalue)
    else: 
        ET.SubElement(parent, _xes_type_name(value), {"key": str(key), "value": _xes_value(value)})

def _xes_type_name(value: Any) -> str:
    if isinstance(value, bool):
        return "boolean"
    if isinstance(value, int) and not isinstance(value, bool):
        return "int"
    if isinstance(value, float):
        return "float"
    if isinstance(value, (dt.datetime, dt.date)):
        return "date"
    return "string"


def _xes_value(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (dt.datetime, dt.date)):
        return value.astimezone().isoformat()
    return str(value)





In [4]:
import pm4py
x = pm4py.deserialize(("event_log", log.to_xes(log_name="sepsis_case")))
for trace in x:
    print(trace)
    for event in trace:
        print('\t', event)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




parsing log, completed traces ::   0%|          | 0/1 [00:00<?, ?it/s]

{'attributes': {'concept:name': 'trace_1', 'name': 'John Maynard', 'age': 68, 'diagnosed': {'value': None, 'children': [('diagnosed', 'diabetes')]}, 'suspected': {'value': None, 'children': [('suspected', 'sepsis')]}}, 'events': [{'concept:name': 'Activity A', 'concept:instance': 'Activity A_1_0', 'time:timestamp': datetime.datetime(2024, 6, 2, 8, 5, tzinfo=datetime.timezone.utc), 'lifecycle:transition': 'start', 'foo': 'bar'}, '..', {'concept:name': 'Activity A', 'concept:instance': 'Activity A_1_0', 'time:timestamp': datetime.datetime(2024, 6, 2, 12, 10, tzinfo=datetime.timezone.utc), 'lifecycle:transition': 'complete', 'resource': 'nurse-2'}]}
	 {'concept:name': 'Activity A', 'concept:instance': 'Activity A_1_0', 'time:timestamp': datetime.datetime(2024, 6, 2, 8, 5, tzinfo=datetime.timezone.utc), 'lifecycle:transition': 'start', 'foo': 'bar'}
	 {'concept:name': 'Activity B', 'concept:instance': 'Activity B_1_4', 'time:timestamp': datetime.datetime(2024, 6, 2, 9, 5, tzinfo=datetime.t

### Experiment Utility

In [5]:
class MockDateTime(dt.datetime):
    now_date = dt.datetime(2020, 1, 2, 3, 4, 2)

    @classmethod
    def set_now_date(cls, new_now_date):
        cls.now_date = new_now_date

    @classmethod
    def now(cls, timezone = None):
        return cls.now_date.replace(tzinfo=timezone)

import contextlib

@contextlib.contextmanager
def mock_time(new_now_date):
    MockDateTime.set_now_date(new_now_date)
    try:
        real_datetime_class = dt.datetime
        dt.datetime = MockDateTime
        yield None
    finally:
        dt.datetime = real_datetime_class


In [6]:
def make_experiment(system_under_eval):
    correct = 0
    total = 0
    def test(bool_test):
        nonlocal correct, total
        total += 1
        if bool_test:
            correct += 1
            print('\033[92m✓', end='')
        else:
            print('\033[91m✗', end='')
            pass # for enabling debugging breakpoints

    def results():
        nonlocal correct, total
        return correct, total

    def run_system(builder):
        log_xes = builder.to_xes()
        current_time = builder.get_current_time()
        result = system_under_eval(log_xes, current_time)
        return result
    
    return test, results, run_system

def test_two_classes(run_system, low_cases, high_cases, test_func):
    low_results = list(map(run_system, low_cases))
    high_results = list(map(run_system, high_cases))
    for low_res in low_results:
        for high_res in high_results:
            test_func(low_res, high_res)

# Domain Definition

In [ ]:

# Activities
ACT_REGISTRATION = "Registration"
ACT_TRIAGE = "Triage"
ACT_TAKE_BLOOD = "Take Blood"
ACT_MEASURE_LEUK = "Measure Leukocytes"
ACT_MEASURE_LACTATE = "Measure Blood Lactate"
ACT_MEASURE_PCT = "Measure Procalcitonin"
ACT_ANTIMICROBIAL = "Administer Antimicrobials"
ACTIVITIES = [ACT_REGISTRATION, ACT_TRIAGE, ACT_TAKE_BLOOD, ACT_MEASURE_LEUK, ACT_MEASURE_LACTATE, ACT_MEASURE_PCT, ACT_ANTIMICROBIAL]

# Case level PVs
SUSPECTED = "suspected"
DIAGNOSED = "diagnosed"
CAUSATIVE_PATHOGEN = "causative pathogen"
MDR_RISK = "multi-drug resistant risk"
# Task level PVs
AGE = "age"
NAME = "name"
INFECTION_SUSPECTED = "infection suspected"
HEART_RATE = "heart rate" # (bpm) ;  SIRS: > 90
BODY_TEMP = "body temperature" # (°C) ; SIRS: >38 | <36
RESP_RATE = "respiratory rate" # (b/min); SIRS: >20
LEUK_COUNT = "leukocyte count" # (M/l) SIRS >12 | <4
LACTATE = "blood lactate" # (mmol/l) 
NUM_SAMPLES = "no. blood samples"
ADMINISTERED_AGENT = "administered agent"

# Antimicrobials
# Antibiotica
VANCOMYCIN = "http://purl.bioontology.org/ontology/SNOMEDCT/372735009" # :is_antiMRSA true .  # No gram-negative coverage
CEFTRIAXONE = "http://purl.bioontology.org/ontology/SNOMEDCT/372670001" # :is_antiMRSA false ; :is_gram_negative true .
PIPERACILLIN = "http://purl.bioontology.org/ontology/SNOMEDCT/372836004" # :is_antiMRSA false ; :is_gram_negative true .
CEFEPIME = "http://purl.bioontology.org/ontology/SNOMEDCT/96048006" # :is_antiMRSA false ; :is_gram_negative true .
METRODINAZOLE = "http://purl.bioontology.org/ontology/SNOMEDCT/372602008" # both not
ANTIBIOTICA = {VANCOMYCIN, CEFTRIAXONE, PIPERACILLIN, CEFEPIME, METRODINAZOLE}
MRSA_COVERAGE = {VANCOMYCIN}
GRAM_NEGATIVE_COVERAGE = {CEFTRIAXONE, PIPERACILLIN, CEFEPIME}
# Antifungals
FLUCONAZOLE = "http://purl.bioontology.org/ontology/SNOMEDCT/387174006"
ECHINOCANDIN = "http://purl.bioontology.org/ontology/SNOMEDCT/373570003"
ANTIFUNGALS = {FLUCONAZOLE, ECHINOCANDIN}
ANTIMICROBIALS = ANTIBIOTICA | ANTIFUNGALS # A simplifcation; there are also, e.g., Antivirals, Antiparasitics, etc. 

# Diseases
SEPSIS = "sepsis"
SEPTIC_SHOCK = "septic shock"
DIABETES = "diabetes"

# MICROBIALS
MRSA = "http://purl.bioontology.org/ontology/SNOMEDCT/115329001" 
ECOLI = "http://purl.bioontology.org/ontology/SNOMEDCT/112283007" # A gram-negative bacterium that is common cause for sepsis
CANDIDA = "http://purl.bioontology.org/ontology/SNOMEDCT/53326005" # A fungus that is a common cause for sepsis
MICROBIALS = {MRSA, ECOLI, CANDIDA}
FUNGI = {CANDIDA}
NON_FUNGI = MICROBIALS - FUNGI

In [ ]:

ACTIVITY_A = "Activity A"
ACTIVITY_B = "Activity B"
ACTIVITY_C = "Activity C"

log = (
    EventLogBuilder()
    .add_trace({NAME : "John Maynard", AGE : 68, DIAGNOSED: {DIABETES}, SUSPECTED : {SEPSIS}})
    .add_task(ACTIVITY_A).start("2024-06-02T08:05:00", {'foo' : "bar"}).complete("2024-06-02T12:10:00", {'resource' : "nurse-2"})
    .add_task(ACTIVITY_C).start("2024-06-02T10:05:00").complete("2024-06-02T10:10:00", {'resource' : "nurse-2"})
    .add_task(ACTIVITY_B).start("2024-06-02T09:05:00").complete("2024-06-02T10:06:00", {'resource' : "nurse-2"})
)

print(log.to_xes(log_name="sepsis_case"))

<?xml version="1.0" encoding="UTF-8"?>
<log xes.version="2.0" xmlns="http://www.xes-standard.org/">
	<string key="concept:name" value="sepsis_case" />
	<trace>
		<string key="concept:name" value="trace_1" />
		<string key="name" value="John Maynard" />
		<int key="age" value="68" />
		<list key="diagnosed">
			<values>
				<string key="diagnosed" value="diabetes" />
			</values>
		</list>
		<list key="suspected">
			<values>
				<string key="suspected" value="sepsis" />
			</values>
		</list>
		<event>
			<string key="concept:name" value="Activity A" />
			<string key="concept:instance" value="Activity A_1_0" />
			<date key="time:timestamp" value="2024-06-02T08:05:00+02:00" />
			<string key="lifecycle:transition" value="start" />
			<string key="foo" value="bar" />
		</event>
		<event>
			<string key="concept:name" value="Activity B" />
			<string key="concept:instance" value="Activity B_1_4" />
			<date key="time:timestamp" value="2024-06-02T09:05:00+02:00" />
			<string key="lifecy

# Guideline Experiments

In [7]:
def base_case_pre_diagnose(infection, heart_rate, body_temp, resp_rate, leuk_count):
    log = (
        EventLogBuilder()
        .add_trace()
        .add_task(ACT_REGISTRATION).start("2024-06-02T07:55:00").complete("2024-06-02T07:59:00", {NAME : "John Maynard", AGE : 68})
        .add_task(ACT_TRIAGE).start("2024-06-02T08:00:00").complete("2024-06-02T08:10:00", {
            INFECTION_SUSPECTED : infection, 
            HEART_RATE : 110 if heart_rate else 78, 
            BODY_TEMP : 39 if body_temp else 37, 
            RESP_RATE : 28 if resp_rate else 19, 
        })
        .add_task(ACT_TAKE_BLOOD).start("2024-06-02T08:02:00").complete("2024-06-02T08:03:00", {NUM_SAMPLES : 3})
        .add_task(ACT_MEASURE_LEUK).start("2024-06-02T08:04:00").complete("2024-06-02T08:08:00", {LEUK_COUNT : 34 if leuk_count else 8})
    )
    return log


log_xes = base_case_pre_diagnose(True, True, False, False, True)
print(log_xes.to_xes())

<?xml version="1.0" encoding="UTF-8"?>
<log xes.version="2.0" xmlns="http://www.xes-standard.org/">
	<string key="concept:name" value="example_log" />
	<trace>
		<string key="concept:name" value="trace_1" />
		<event>
			<string key="concept:name" value="Registration" />
			<string key="concept:instance" value="Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T07:55:00+02:00" />
			<string key="lifecycle:transition" value="start" />
		</event>
		<event>
			<string key="concept:name" value="Registration" />
			<string key="concept:instance" value="Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T07:59:00+02:00" />
			<string key="lifecycle:transition" value="complete" />
			<string key="name" value="John Maynard" />
			<int key="age" value="68" />
		</event>
		<event>
			<string key="concept:name" value="Triage" />
			<string key="concept:instance" value="Triage_1_2" />
			<date key="time:timestamp" value="2024-06-02T08:00:00+02:00" />
			<string k

## Blood Lactate

### "For adults suspected of having sepsis, we suggest measuring blood lactate." ...
... Suspicion of sepsis is governed by a suspected infection and the presence of at least two of the SIRS criteria (i) body temperature outside of [36°C;38°C], (ii) heartrate > 90bpm, (iii) respiratory rate > 20 breaths/minute, and (iv) leukocyte count outside of [4M/l;12M/l]

In [8]:
def measure_lactate_base_experiment(system_under_eval):
    test, results, run_system = make_experiment(system_under_eval)
    test_two_classes(run_system, [
        base_case_pre_diagnose(False, False, False, False, False), # baseline
        base_case_pre_diagnose(False, False, True, True, False), # no infection
        base_case_pre_diagnose(True, True, False, False, False) # not sufficient SIRS
    ], [
        base_case_pre_diagnose(True, True, True, True, True), # baseline
        base_case_pre_diagnose(True, False, True, True, False), # body temp and resp
        base_case_pre_diagnose(True, True, False, False, True) # heart rate and leuk
    ], lambda low_res, high_res: test(high_res.get(ACT_MEASURE_LACTATE, 0) > low_res.get(ACT_MEASURE_LACTATE, 0)))
    return results()

### "For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum lactate in patients with elevated lactate levels over not using serum lactate." ...
...  "Measure the lactate of patients [...] again [every] 2-4 hours if the [last] result is greater than 2mmol/L"

In [9]:
def measure_lactate_time_experiment(system_under_eval):
    test, results, run_system = make_experiment(system_under_eval)

    # Case 1: Check that it's more important than a non-urgent measurement
    case = base_case_pre_diagnose(True, False, True, True, False)
    result = run_system(case)
    test(result.get(ACT_MEASURE_LACTATE, 0) > result.get(ACT_MEASURE_PCT, 0)) # TODO: Within situation measurement

    # Case 2: Check that it's repeated if two hours have passed and the last measurement was > 2 mmol/L
    test_two_classes(run_system, [
        (base_case_pre_diagnose(True, True, True, False, False) # Has been long enough ago but not high enough
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 1.6})
            .set_current_time("2024-06-02T10:20:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been high but not long enough ago
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .set_current_time("2024-06-02T09:15:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been long enough ago but not high enough
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 1.8})
            .set_current_time("2024-06-02T14:00:00"))
    ], [
        (base_case_pre_diagnose(True, True, True, False, False) # Has been high a while ago
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .set_current_time("2024-06-02T10:20:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been continuously high
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 2.2})
            .set_current_time("2024-06-02T14:00:00"))
    ], lambda low_res, high_res: test(high_res.get(ACT_MEASURE_LACTATE, 0) > low_res.get(ACT_MEASURE_LACTATE, 0)))

    return results()


## Antimicrobials

In [10]:
def triaged_patient_base(case_attr=dict()): 
    log = (
        EventLogBuilder()
        .add_trace(case_attr)
        .add_task(ACT_REGISTRATION).start("2024-06-02T07:55:00").complete("2024-06-02T07:59:00", {NAME : "John Maynard", AGE : 68})
        .add_task(ACT_TRIAGE).start("2024-06-02T08:00:00").complete("2024-06-02T08:10:00", {
            INFECTION_SUSPECTED : True, 
            HEART_RATE : 110, 
            BODY_TEMP : 37, 
            RESP_RATE : 28
        })
            .add_task(ACT_TAKE_BLOOD).start("2024-06-02T08:02:00").complete("2024-06-02T08:03:00", {NUM_SAMPLES : 3})
            .add_task(ACT_MEASURE_LEUK).start("2024-06-02T08:04:00").complete("2024-06-02T08:08:00", {LEUK_COUNT : 34})
        .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:11:00").complete("2024-06-02T08:16:00", {LACTATE : 2.4})
    )
    return log


log_xes = triaged_patient_base({DIAGNOSED : [DIABETES, SEPSIS, SEPTIC_SHOCK]})
print(log_xes.to_xes())

<?xml version="1.0" encoding="UTF-8"?>
<log xes.version="2.0" xmlns="http://www.xes-standard.org/">
	<string key="concept:name" value="example_log" />
	<trace>
		<string key="concept:name" value="trace_1" />
		<list key="diagnosed">
			<values>
				<string key="diagnosed" value="diabetes" />
				<string key="diagnosed" value="sepsis" />
				<string key="diagnosed" value="septic shock" />
			</values>
		</list>
		<event>
			<string key="concept:name" value="Registration" />
			<string key="concept:instance" value="Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T07:55:00+02:00" />
			<string key="lifecycle:transition" value="start" />
		</event>
		<event>
			<string key="concept:name" value="Registration" />
			<string key="concept:instance" value="Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T07:59:00+02:00" />
			<string key="lifecycle:transition" value="complete" />
			<string key="name" value="John Maynard" />
			<int key="age" value="68" /

### "For adults with sepsis or septic shock at high risk of [methicillin-resistant Staphylococcus aureus (MRSA)], ... 
... we recommend using empiric antimicrobials with MRSA coverage over using antimicrobials without MRSA coverage [and] at low risk of MRSA, we suggest against using empiric antimicrobials with MRSA coverage, as compared with using antimicrobials without MRSA coverage."

In [11]:
def antimicrobials_MRSA_experiment(system_under_eval):
    test, results, run_system = make_experiment(system_under_eval)
    f_A = run_system(triaged_patient_base({DIAGNOSED : [SEPSIS]}))
    f_B = run_system(triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : MRSA}))
    for mrsa_cover in MRSA_COVERAGE:
        test(f_B.get((ACT_ANTIMICROBIAL, mrsa_cover), 0) > f_A.get((ACT_ANTIMICROBIAL, mrsa_cover), 0))
        test(f_A.get((ACT_ANTIMICROBIAL, mrsa_cover), 0) < 0) # TODO Fixed number
        for other in ANTIMICROBIALS - MRSA_COVERAGE:
            test(f_B.get((ACT_ANTIMICROBIAL, mrsa_cover), 0) > f_B.get((ACT_ANTIMICROBIAL, other), 0))

    return results()

### "For adults with sepsis or septic shock and high risk for multidrug-resistant (MDR) organisms, ...
... we suggest using 2 antimicrobials with gram-negative coverage for empiric treatment over 1 gram-negative agent [and, for] low risk for multidrug-resistant (MDR) organisms, we suggest against using 2 gram-negative agents for empiric treatment, as compared to 1 gram-negative agent. [...] We suggest against using double gram-negative coverage once the causative pathogen and the susceptibilities are known." 

In [12]:
def antimicrobials_MDR_experiment(system_under_eval):
    test, results, run_system = make_experiment(system_under_eval)
    test_two_classes(run_system, [
        triaged_patient_base({DIAGNOSED : [SEPSIS]}), # Baseline for no double treatment, no increased risk
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}) # Already administered two gram-negative antibiotics
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:15:00", {ADMINISTERED_AGENT : PIPERACILLIN}).complete("2024-06-02T09:30:00"),
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True, CAUSATIVE_PATHOGEN : ECOLI}) # Only administered one gram-negative antibiotic, but causative pathogen is known
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
    ], [
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}), # Baseline for double treatment, increased risk, no antibiotics administered so far
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}) # Only one gram-negative antibiotic administered so far (although twice)
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T09:00:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T09:30:00")
    ], lambda low_res, high_res: test(high_res.get((ACT_ANTIMICROBIAL, CEFEPIME), 0) > low_res.get((ACT_ANTIMICROBIAL, CEFEPIME), 0)))
    return results()

### "For adults with sepsis or septic shock at high risk of fungal infection, ... 
... we suggest using empiric antifungal therapy over no antifungal therapy [and] at low risk of fungal infection, we suggest against empiric use of antifungal therapy."

In [13]:
def antimicrobials_Fungi_experiment(system_under_eval):
    test, results, run_system = make_experiment(system_under_eval)

    f_A = run_system(triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : MRSA}))
    f_B = run_system(triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : CANDIDA})) # TODO could add intermedate case with unknown pathogen

    for antifungal in ANTIFUNGALS:
        test(f_B.get((ACT_ANTIMICROBIAL, antifungal), 0) > f_A.get((ACT_ANTIMICROBIAL, antifungal), 0))
        test(f_A.get((ACT_ANTIMICROBIAL, antifungal), 0) < 0) # TODO Fixed number
        for other in ANTIMICROBIALS - ANTIFUNGALS:
            test(f_B.get((ACT_ANTIMICROBIAL, antifungal), 0) > f_B.get((ACT_ANTIMICROBIAL, other), 0))

    return results()

# Karibdis

#### Experiment Bridge

In [14]:
import os
import sys
karibdis_path = os.path.abspath('../../src')
print(karibdis_path)
sys.path.insert(0, karibdis_path)

c:\unsynchronized\projects\.Graph Workbench\Workbench\karibdis_project\karibdis\src


In [15]:
%load_ext autoreload
%autoreload 2
from karibdis.KnowledgeGraphBPMS import KnowledgeGraphBPMS
from karibdis.KnowledgeImporter import OnlineEventImporter
from rdflib import Graph, RDF
from karibdis.utils import BASE_PROCESS_ONTOLOGY as BPO

In [16]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from rdflib import Graph, Variable

In [17]:
def karibdis_valuations():
    bpms = KnowledgeGraphBPMS() 
    bpms.pkg += Graph().parse('additional_knowledge.ttl', format='turtle')
    bpms.pkg += Graph().parse('snomed_antimicrobials.ttl', format='turtle')
    bpms.engine.deduce_chained()

    pre_loader = OnlineEventImporter(bpms.pkg)
    activity_map = {}
    for activity in ACTIVITIES:
        activity_uri = pre_loader.activity_node(activity)
        bpms.pkg.add((activity_uri, RDF.type, BPO.Activity))
        activity_map[activity_uri] = activity
    print(f'Mapping Activity URIs: {activity_map}')

    def karibdis_load_log_prefix(log_xes, bpms):
        x = xes_importer.deserialize(log_xes, {"show_progress_bar": False})
        loader = OnlineEventImporter(bpms.pkg)
        for trace in x:
            # print(trace)
            loader.case_attributes = trace._attributes.keys() - {'concept:name'}
            for case_attr in loader.case_attributes:
                trace[0][case_attr] = trace._attributes[case_attr]
            for event in trace:
                event['case:concept:name'] = trace._attributes['concept:name']
                for key, value in event.items():
                    if hasattr(value, "items") and value['children']:
                        event[key] = list(zip(*value['children']))[1]
                # print(event)
                loader.translate_event(event) # TODO: Potentially use existing load-whole-log-function
        loader.load()

    def get_valuations(log_xes, current_time):
        triples_before = list(bpms.pkg)
        try:
            with mock_time(current_time):
                karibdis_load_log_prefix(log_xes, bpms)
                bpms.engine.deduce_chained()
                decision = next(bpms.engine.open_decisions())
                valuation, options, reasoning = list(zip(*decision.get_top_k_results())) # TODO translate back from URIs
        finally:
            bpms.pkg -= bpms.pkg
            bpms.pkg += triples_before
        print(valuation, options)
        def translate_option(option):
            option_key, effect = option
            result = activity_map.get(option_key['activity'])
            if len(option_key.keys()) > 1:
                result = tuple([result, *map(lambda key: option_key[key].toPython(), sorted(option_key.keys() - {Variable('activity')}))])
            return result 
        return dict(zip(map(translate_option, options), valuation))

    return get_valuations

### Experiments

In [21]:
measure_lactate_base_experiment(karibdis_valuations())

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCriteria2OrMore'), rdflib.term.Literal('false', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))),

(9, 9)

In [22]:
measure_lactate_time_experiment(karibdis_valuations())

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritTachypnea'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (r

(7, 7)

In [23]:
antimicrobials_MRSA_experiment(karibdis_valuations())

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritHeartRate'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (r

(8, 8)

In [24]:
antimicrobials_MDR_experiment(karibdis_valuations())

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritHeartRate'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (r

(6, 6)

In [25]:
antimicrobials_Fungi_experiment(karibdis_valuations())

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}


{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritHeartRate'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (rdflib.term.URIRef('http://example.org/Task_trace_1_6'), rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/partOf'), rdflib.term.URIRef('http://example.org/Case_trace_1')), (rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritTachypnea'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritTemperature'), rdflib.term.Literal('false', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (rdflib.term.URIRef('http://example.org/Task_trace_1_6')

(14, 14)

# Workbench

In [26]:
raise 'WIP, Do not enter'

TypeError: exceptions must derive from BaseException

In [ ]:
print(make_experiment(karibdis_valuations())[2](base_case_pre_diagnose(True, False, True, True, False)))

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritTachypnea'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (r

In [ ]:
print(make_experiment(karibdis_valuations())[2](base_case_pre_diagnose(True, True, True, True, True)))

{'knowledge_updated': True}
Mapping Activity URIs: {rdflib.term.URIRef('http://example.org/Activity_Registration'): 'Registration', rdflib.term.URIRef('http://example.org/Activity_Triage'): 'Triage', rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'): 'Take Blood', rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'): 'Measure Leukocytes', rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'): 'Measure Blood Lactate', rdflib.term.URIRef('http://example.org/Activity_Measure%20Procalcitonin'): 'Measure Procalcitonin', rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials'): 'Administer Antimicrobials'}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritHeartRate'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (r

In [ ]:
bpms2 = KnowledgeGraphBPMS() 
bpms2.pkg += Graph().parse('additional_knowledge.ttl', format='turtle')
bpms2.pkg += Graph().parse('snomed_antimicrobials.ttl', format='turtle')
pre_loader = OnlineEventImporter(bpms2.pkg)
activity_map = {}
for activity in ACTIVITIES:
    activity_uri = pre_loader.activity_node(activity)
    bpms2.pkg.add((activity_uri, RDF.type, BPO.Activity))
    activity_map[activity_uri] = activity
bpms2.engine.deduce_chained()

# case = (triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : MRSA}) # Has been long enough ago but not high enough
#             .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
#             .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 2.2}))            .set_current_time("2024-06-02T10:20:00")

case = (triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}) # Already administered two gram-negative antibiotics
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:15:00", {ADMINISTERED_AGENT : PIPERACILLIN}).complete("2024-06-02T09:30:00"))

x = xes_importer.deserialize(case.to_xes(), {"show_progress_bar": False})
loader = OnlineEventImporter(bpms2.pkg)
for trace in x:
    print(trace)
    loader.case_attributes = trace._attributes.keys() - {'concept:name'}
    for case_attr in loader.case_attributes:
        trace[0][case_attr] = trace._attributes[case_attr]
    for event in trace:
        event['case:concept:name'] = trace._attributes['concept:name']
        for key, value in event.items():
            if hasattr(value, "items") and value['children']:
                event[key] = list(zip(*value['children']))[1]
        # print(event)
        loader.translate_event(event) # TODO: Potentially use existing load-whole-log-function
loader.load()
bpms2.engine.deduce_chained()
pkg = bpms2.pkg

{'knowledge_updated': True}
{'attributes': {'concept:name': 'trace_1', 'diagnosed': {'value': None, 'children': [('diagnosed', 'sepsis')]}, 'multi-drug resistant risk': True}, 'events': [{'concept:name': 'Registration', 'concept:instance': 'Registration_1_0', 'time:timestamp': datetime.datetime(2024, 6, 2, 7, 55, tzinfo=datetime.timezone.utc), 'lifecycle:transition': 'start'}, '..', {'concept:name': 'Administer Antimicrobials', 'concept:instance': 'Administer Antimicrobials_1_12', 'time:timestamp': datetime.datetime(2024, 6, 2, 9, 30, tzinfo=datetime.timezone.utc), 'lifecycle:transition': 'complete'}]}
{'knowledge_updated': True}
{'knowledge_updated': True, 'removed': set(), 'added': {(rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef('http://example.org/ProcessValue_SIRSCritHeartRate'), rdflib.term.Literal('true', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#boolean'))), (rdflib.term.URIRef('http://example.org/Case_trace_1'), rdflib.term.URIRef

In [ ]:
list(pkg.subjects(predicate=RDF.type, object=BPO.Case))[0]

rdflib.term.URIRef('http://example.org/Case_trace_1')

In [ ]:
next(bpms2.engine.open_decisions()).get_top_k_results()

[(10,
  ({rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate')},
   [(rdflib.term.URIRef('http://example.org/Task_trace_1_8'),
     rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/instanceOf'),
     rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'))]),
  ['Lactate levels should be measured in patients suspected of having sepsis.',
   'Lactate levels should be re-measured every 2-4 hours if the last result is greater than 2mmol/L. Last measurement 2.2 mmol/L at 2024-06-02T11:15:00+00:00.']),
 (0,
  ({rdflib.term.Variable('anti'): rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/373219008'), rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials')},
   [(rdflib.term.URIRef('http://example.org/Task_trace_1_8'),
     rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/instanceOf'),
     rdflib.term.URIRef('h

In [ ]:
list(bpms2.pkg.query(''' 
    SELECT $this
    WHERE {
        $this rdfs:subClassOf+ <http://purl.bioontology.org/ontology/SNOMEDCT/373219008> . # Antifungal
    }
'''))

[(rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/387174006'),),
 (rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/373570003'),)]

In [ ]:
from karibdis.utils import draw_graph
draw_graph(bpms2.pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [ ]:
list(bpms2.pkg.query(''' 
    PREFIX : <http://www.example.org/ontology#>
    PREFIX log: <http://www.example.org/ontology1#>
    SELECT  ?measuring ?lactate_level ?maxCompletedAt
    WHERE {
        {
            SELECT (MAX(?completedAt) AS ?maxCompletedAt) WHERE {
                ?any_measuring :partOf ?case .
                ?any_measuring :completedAt ?completedAt .
                ?any_measuring log:ProcessValue_blood%20lactate ?lactate_level .
            } 
        }
        ?measuring :completedAt ?maxCompletedAt .
        ?measuring :partOf ?case .
        ?measuring log:ProcessValue_blood%20lactate ?lactate_level .
    }
'''))

[]

In [ ]:
list(bpms2.pkg.query(''' 
    PREFIX : <http://www.example.org/ontology#>
    PREFIX log: <http://www.example.org/ontology1#>
    SELECT  ?measuring ?lactate_level ?maxCompletedAt
    WHERE {
        {
            SELECT (MAX(?completedAt) AS ?maxCompletedAt) WHERE {
                ?any_measuring :partOf ?case .
                ?any_measuring :completedAt ?completedAt .
                ?any_measuring log:ProcessValue_blood%20lactate ?lactate_level .
            }
        }
        ?measuring :completedAt ?maxCompletedAt .
        ?measuring :partOf ?case .
        ?measuring log:ProcessValue_blood%20lactate ?lactate_level .
        FILTER((NOW() - "PT2H"^^xsd:duration) > ?maxCompletedAt) .
        FILTER(?lactate_level > 2.0) .
    }
'''))

[]

In [ ]:
draw_graph(bpms2.pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [ ]:
bpms2 = KnowledgeGraphBPMS() 

In [ ]:
from rdflib.plugins.sparql.sparql import QueryContext
QueryContext().now

datetime.datetime(2026, 8, 26, 15, 31, 25, 114864, tzinfo=datetime.timezone.utc)

In [ ]:
Graph().query

<bound method Graph.query of <Graph identifier=N08ea6e3737574840bf8fc1ab46348ce3 (<class 'rdflib.graph.Graph'>)>>

In [ ]:
import datetime as dt
real_datetime_class = dt.datetime

In [ ]:
dt.datetime = real_datetime_class

In [ ]:
real_datetime_class(2010, 1, 1, 1, 1)

datetime.datetime(2010, 1, 1, 1, 1)

In [ ]:
from rdflib import XSD, Graph, URIRef, Literal
from rdflib.plugins.sparql.operators import register_custom_function
import rdflib.plugins.sparql.operators

with mock_time(dt.datetime(2024, 6, 2, 12, 0)):
    print(list(Graph().query(''' 
        SELECT ?now
        WHERE {
            BIND(NOW() AS ?now)
        }
        '''
    ))[0][0].toPython())
print(list(Graph().query(''' 
    SELECT ?now
    WHERE {
        BIND(NOW() AS ?now)
    }
    '''
))[0][0].toPython())

2024-06-02 12:00:00+00:00


IndexError: list index out of range

In [ ]:
list(Graph().query(''' 
    PREFIX : <http://www.example.org/ontology#>
    PREFIX log: <http://www.example.org/ontology1#>
    SELECT  $this (5 AS ?value)
    WHERE {
        $this :instanceOf__hypothetical log:Activity_Measure%20Blood%20Lactate .
        $this :partOf ?case .
        {
            SELECT (MAX(?completedAt) AS ?maxCompletedAt) WHERE {
                ?any_measuring :partOf ?case .
                ?any_measuring :completedAt ?completedAt .
                ?any_measuring log:ProcessValue_blood%20lactate ?lactate_level .
            }
        }
        ?measuring :completedAt  ?maxCompletedAt .
        ?measuring :partOf ?case .
        ?measuring log:ProcessValue_blood%20lactate ?lactate_level .
        FILTER((NOW() - "PT2H"^^xsd:duration) > ?timestamp) .
        FILTER(?lactate_level > 2.0) .
    }
'''))

[]

In [ ]:
from karibdis.utils import draw_graph


In [ ]:
x = pm4py.deserialize(("event_log", base_case_pre_diagnose(True, True, True, True, True)))
bpms2 = KnowledgeGraphBPMS() 
bpms2.pkg += Graph().parse('additional_knowledge.ttl', format='turtle')
karibdis_valuations(bpms2)
bpms2.engine.deduce_chained()
loader = OnlineEventImporter(bpms2.pkg)
for trace in x:
    print(trace)
    for event in trace:
        event['case:concept:name'] = trace._attributes['concept:name']
        for key, value in event.items():
            if hasattr(value, "items") and value['children']:
                event[key] = list(zip(*value['children']))[1]
        # print(event)
        loader.translate_event(event)
loader.load()
print(bpms2.engine.deduce_chained())
draw_graph(loader.addition_graph)

In [ ]:
next(bpms2.engine.open_decisions()).subject

rdflib.term.URIRef('http://example.org/Task_trace_1_5')

In [ ]:
next(bpms2.engine.open_decisions()).get_top_k_results()

[(5,
  rdflib.term.URIRef('http://example.org/Activity_Measure%20Blood%20Lactate'),
  ['Lactate levels should be measured in patients suspected of having sepsis.']),
 (0, rdflib.term.URIRef('http://example.org/Activity_Take%20Blood'), []),
 (0, rdflib.term.URIRef('http://example.org/Activity_Triage'), []),
 (0,
  rdflib.term.URIRef('http://example.org/Activity_Measure%20Leukocytes'),
  []),
 (0, rdflib.term.URIRef('http://example.org/Activity_Registration'), [])]

In [ ]:
from rdflib import URIRef
bpms2.pkg.remove((URIRef('http://example.org/Task_trace_1_5'), BPO.instanceOf + '__hypothetical', URIRef('http://example.org/Activity_Measure%20Blood%20Lactate')))

<Graph identifier=Nf4148ce92b584b1fb86df79bfacca12e (<class 'karibdis.ProcessKnowledgeGraph.ProcessKnowledgeGraph'>)>

In [ ]:
list(bpms2.pkg.query('''
    SELECT  $this (5 AS ?value)
    WHERE {
        $this :instanceOf__hypothetical log:Activity_Measure%20Blood%20Lactate .
        $this :partOf / :ProcessValue_suspected <http://purl.obolibrary.org/obo/MONDO_1040015> . # Sepsis
    }
'''))

[]

In [ ]:
from urllib.request import urlopen, Request
from urllib.parse import quote
import json

baseUrl = 'https://browser.ihtsdotools.org/snowstorm/snomed-ct'
edition = 'MAIN'
version = '2019-07-31'

# IMPORTANT! You must update this user agent to avoid having your IP banned for 24 hours.
# Replace with a contact email so that we can contact you if your script causes excessive load on the public server
# For example: user_agent = 'example@example.com'
user_agent = 'leon.bein@tum.de'

def urlopen_with_header(url):
    # adds User-Agent header otherwise urlopen on its own gets an IP blocked response
    req = Request(url)
    req.add_header('User-Agent', user_agent)
    return urlopen(req)

#Prints fsn of a concept
def getConceptById(id):
    url = baseUrl + '/browser/' + edition + '/' + version + '/concepts/' + id
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    print (data['fsn']['term'])

#Prints description by id
def getDescriptionById(id):
    url = baseUrl + '/' + edition + '/' + version + '/descriptions/' + id
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    print (data['term'])

#Prints number of concepts with descriptions containing the search term
def getConceptsByString(searchTerm):
    url = baseUrl + '/browser/' + edition + '/' + version + '/concepts?term=' + quote(searchTerm) + '&activeFilter=true&offset=0&limit=50'
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    print (data['total'])

#Prints number of descriptions containing the search term with a specific semantic tag
def getDescriptionsByStringFromProcedure(searchTerm, semanticTag):
    url = baseUrl + '/browser/' + edition + '/' + version + '/descriptions?term=' + quote(searchTerm) + '&conceptActive=true&semanticTag=' + quote(semanticTag) + '&groupByConcept=false&searchMode=STANDARD&offset=0&limit=50'
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    print (data['totalElements'])
    
 #Prints snomed code for searched disease or symptom
def getSnomedCodeSimilar(searchTerm):
    url = baseUrl + '/browser/' + edition + '/' + version + '/descriptions?term=' + quote(searchTerm) + '&conceptActive=true&groupByConcept=false&searchMode=STANDARD&offset=0&limit=50'
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    for term in data['items']:
      if searchTerm in term['term']:
        print("{} : {}".format(term['term'], term['concept']['conceptId']))
 
def getSnomedCode(searchTerm):
    url = baseUrl + '/browser/' + edition + '/' + version + '/descriptions?term=' + quote(searchTerm) + '&conceptActive=true&groupByConcept=false&searchMode=STANDARD&offset=0&limit=50'
    response = urlopen_with_header(url).read()
    data = json.loads(response.decode('utf-8'))

    for term in data['items']:
      if searchTerm == term['term']:
        print("{} : {}".format(term['term'], term['concept']['conceptId']))


HTTPError: HTTP Error 503: Service Temporarily Unavailable

In [ ]:

getConceptById('109152007')

HTTPError: HTTP Error 503: Service Temporarily Unavailable

In [ ]:
g = Graph().parse('http://purl.bioontology.org/ontology/SNOMEDCT', format='ttl')

https://bioportal.bioontology.org/ontologies/!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd" does not look like a valid URI, trying to serialize this will break.
html xmlns="http://www.w3.org/1999/xhtml" class="h-100" does not look like a valid URI, trying to serialize this will break.


BadSyntax: at line 4 of <>:
Bad syntax (expected '.' or '}' or ']' at end of statement) at ^ in:
"...b'xmlns="http://www.w3.org/1999/xhtml" class="h-100">\n<head>\n\t'^b'<script async src="https://www.googletagmanager.com/gtag/js?'..."

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint = SPARQLWrapper("http://bioportal.bioontology.org/ontologies/SNOMEDCT")

endpoint.setQuery("""

SELECT ?concept ?label
FROM <http://bioportal.bioontology.org/ontologies/SNOMEDCT>
WHERE {
    <http://purl.bioontology.org/ontology/SNOMEDCT/68322007> rdfs:subclassof ?concept .
    ?concept <http://www.w3.org/2004/02/skos/core#prefLabel> ?label .
}
LIMIT 100
""")

endpoint.setReturnFormat(JSON)

results = endpoint.query().convert()

for row in results["results"]["bindings"]:
    print(
        row["concept"]["value"],
        row["label"]["value"]
    )

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))